In [22]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import TypedDict, Annotated , Literal
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
import operator

In [23]:
generator_llm = ChatOllama(
    model="llama3",
    temperature=0
)

In [24]:
evaluation_llm = ChatOllama(
    model="llama3",
    temperature=0
)

In [25]:
optimisation_llm = ChatOllama(
    model="llama3",
    temperature=0
)

In [26]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [27]:
structured_evaluator_llm = evaluation_llm.with_structured_output(TweetEvaluation)

In [28]:
#define state 

class TweetState(TypedDict):

    topic : str
    tweet : str
    evaluation_generate = Literal["Approved", "Needs Improvement"]
    feedback : str
    iteration : int
    max_iteration : int

    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]

In [29]:
def generate_tweet(state : TweetState):
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    response = generator_llm.invoke(messages)
    return {'tweet': response, 'tweet_history': [response]}

In [9]:
def evaluate_tweet(state : TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]
    
    response = structured_evaluator_llm.invoke(messages)

    return {'evaluation':response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [30]:
def optimize_tweet(state: TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = optimisation_llm.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [34]:
def route_evaluation(state: TweetState):

    evaluation = state.get("evaluation")

    if evaluation == "approved" or state["iteration"] >= state["max_iteration"]:
        return "approved"
    else:
        return "needs_improvement"

In [35]:
#define graph
graph = StateGraph(TweetState)

#add nodes to your graph
graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimiser', optimize_tweet)

#add edges to your graph
graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement':'optimiser'})
graph.add_edge('optimiser','evaluate')

workflow = graph.compile()

In [36]:
initial_state = {
    "topic": "srhberhb",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)

In [37]:
result

{'topic': 'srhberhb',
 'tweet': 'Here\'s a rewritten tweet that\'s even punchier and concise:\n\n"Cracked the code: \'srhberhb\' = adulting anxiety 😂 #adultingfail"\n\nI added a 😂 emoji to make the tweet more engaging and shareable, as suggested in the feedback. The emoji adds a playful touch and helps convey the humor and relatability of the tweet.',
 'feedback': "This tweet is a great example of concise and punchy humor. The use of the 'cracked the code' phrase is clever and sets up the punchline nicely. The addition of the 😂 emoji helps to convey the humor and relatability of the tweet, making it more engaging and shareable. The only potential area for improvement is that the joke may not be entirely new or original, but it's still a well-crafted and effective tweet.",
 'iteration': 5,
 'max_iteration': 5,
 'tweet_history': [AIMessage(content='"Just spent 10 minutes trying to remember what \'srhberhb\' means and I\'m starting to think it\'s just a secret code for \'I have no idea wh

In [38]:
for tweet in result['tweet_history']:
    print(tweet)

content='"Just spent 10 minutes trying to remember what \'srhberhb\' means and I\'m starting to think it\'s just a secret code for \'I have no idea what I\'m doing with my life\' #adultingfail"' additional_kwargs={} response_metadata={'model': 'llama3', 'created_at': '2026-03-05T09:08:24.5620023Z', 'done': True, 'done_reason': 'stop', 'total_duration': 41426921700, 'load_duration': 10602755200, 'prompt_eval_count': 98, 'prompt_eval_duration': 12657984900, 'eval_count': 48, 'eval_duration': 17998678600, 'logprobs': None, 'model_name': 'llama3', 'model_provider': 'ollama'} id='lc_run--019cbd40-bf9c-7060-8860-6695f5d8544f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 98, 'output_tokens': 48, 'total_tokens': 146}
Here's a rewritten tweet that's even punchier and concise:

"I think I just cracked the code: 'srhberhb' = 'I have no idea what I'm doing with my life' #adultingfail"

This tweet maintains the humor and relatability of the original while being even more co